# Module 1 — Anomaly Detection (Isolation Forest)
### AI-Powered Student LMS | Phase 3
**Goal:** Detect students with abnormal academic behaviour early — before exams.

**Algorithm:** Isolation Forest — works by isolating anomalies using random splits. Students who are isolated quickly = anomalous = at-risk.

---

In [ ]:
# ── CELL 1 ── Imports ──────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
import joblib

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.decomposition import PCA

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

# Paths
BASE   = r'C:\Users\DELL\Desktop\AI_Student_LMS'
DATA   = os.path.join(BASE, 'data', 'raw', 'data.csv')
MODELS = os.path.join(BASE, 'models')
ASSETS = os.path.join(BASE, 'assets')
os.makedirs(MODELS, exist_ok=True)
os.makedirs(ASSETS, exist_ok=True)

print('✅ All imports done!')

In [ ]:
# ── CELL 2 ── Load & Prepare Data ─────────────────────────────
df = pd.read_csv(DATA, sep=';')
print(f'Dataset loaded: {df.shape}')

# Encode target label
le = LabelEncoder()
df['Target_enc'] = le.fit_transform(df['Target'])  # Dropout=0, Enrolled=1, Graduate=2
print('\nTarget encoding:')
for i, cls in enumerate(le.classes_):
    print(f'  {i} = {cls}')

# Select numerical features for anomaly detection
FEATURES = [
    'Age at enrollment',
    'Admission grade',
    'Curricular units 1st sem (enrolled)',
    'Curricular units 1st sem (approved)',
    'Curricular units 1st sem (grade)',
    'Curricular units 2nd sem (enrolled)',
    'Curricular units 2nd sem (approved)',
    'Curricular units 2nd sem (grade)',
    'Curricular units 1st sem (evaluations)',
    'Curricular units 2nd sem (evaluations)',
    'Previous qualification (grade)',
    'Scholarship holder',
    'Debtor',
    'Tuition fees up to date',
    'Gender',
]

# Keep only columns that exist in dataset
FEATURES = [f for f in FEATURES if f in df.columns]
print(f'\n✅ Features selected: {len(FEATURES)}')
print(FEATURES)

X = df[FEATURES].copy()
y_true = df['Target'].copy()
print(f'\nFeature matrix shape: {X.shape}')

In [ ]:
# ── CELL 3 ── Scale Features ───────────────────────────────────
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print('✅ Features scaled with StandardScaler')
print(f'Mean (first 3 features): {X_scaled[:, :3].mean(axis=0).round(4)}')
print(f'Std  (first 3 features): {X_scaled[:, :3].std(axis=0).round(4)}')

In [ ]:
# ── CELL 4 ── Train Isolation Forest ──────────────────────────
# contamination = estimated proportion of anomalies in data
# Dropout rate ~32% but true anomalies are a subset — use 0.15
iso_forest = IsolationForest(
    n_estimators=200,       # More trees = more stable
    contamination=0.15,     # 15% of students flagged as anomalous
    max_samples='auto',
    random_state=42,
    n_jobs=-1               # Use all CPU cores
)

iso_forest.fit(X_scaled)

# Predictions: -1 = anomaly (at-risk), 1 = normal
df['anomaly_flag'] = iso_forest.predict(X_scaled)

# Anomaly score: lower = more anomalous
df['anomaly_score'] = iso_forest.decision_function(X_scaled)

# Convert to risk label
df['risk_label'] = df['anomaly_flag'].map({-1: '⚠️ AT-RISK', 1: '✅ NORMAL'})

n_atrisk = (df['anomaly_flag'] == -1).sum()
n_normal = (df['anomaly_flag'] == 1).sum()

print('✅ Isolation Forest trained!')
print(f'\n🔴 AT-RISK students flagged : {n_atrisk} ({n_atrisk/len(df)*100:.1f}%)')
print(f'🟢 NORMAL students          : {n_normal} ({n_normal/len(df)*100:.1f}%)')

In [ ]:
# ── CELL 5 ── Validate: Do anomalies match actual dropouts? ───
print('='*55)
print('VALIDATION: Anomaly Flag vs Actual Target')
print('='*55)
crosstab = pd.crosstab(df['risk_label'], df['Target'], margins=True)
print(crosstab)

# What % of flagged students actually dropped out?
atrisk_df  = df[df['anomaly_flag'] == -1]
dropout_in_atrisk = (atrisk_df['Target'] == 'Dropout').sum()
pct = dropout_in_atrisk / len(atrisk_df) * 100

print(f'\n📊 Of all AT-RISK flagged students:')
print(f'   → {dropout_in_atrisk} are actual Dropouts ({pct:.1f}%)')
print(f'   → This validates our model is catching real at-risk students!')

In [ ]:
# ── CELL 6 ── PLOT 1: Anomaly Score Distribution ──────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Score distribution
colors = df['anomaly_flag'].map({-1: '#E74C3C', 1: '#2ECC71'})
axes[0].scatter(range(len(df)), df['anomaly_score'],
                c=colors, alpha=0.4, s=8)
axes[0].axhline(0, color='black', linestyle='--', linewidth=1.5,
                label='Decision boundary')
axes[0].set_title('Anomaly Score per Student', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Student Index')
axes[0].set_ylabel('Anomaly Score (lower = more anomalous)')
axes[0].legend()

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#E74C3C', label='AT-RISK'),
                   Patch(facecolor='#2ECC71', label='NORMAL')]
axes[0].legend(handles=legend_elements)

# Right: Score histogram by actual target
for status, color in zip(['Dropout', 'Enrolled', 'Graduate'],
                          ['#E74C3C', '#F39C12', '#2ECC71']):
    subset = df[df['Target'] == status]['anomaly_score']
    axes[1].hist(subset, bins=40, alpha=0.6, color=color, label=status)
axes[1].axvline(0, color='black', linestyle='--', linewidth=1.5)
axes[1].set_title('Score Distribution by Actual Status', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Anomaly Score')
axes[1].set_ylabel('Number of Students')
axes[1].legend()

plt.suptitle('Isolation Forest — Anomaly Detection Results', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(ASSETS, 'anomaly_01_score_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✅ Plot saved!')

In [ ]:
# ── CELL 7 ── PLOT 2: PCA 2D Visualization ────────────────────
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(10, 7))
color_map = df['anomaly_flag'].map({-1: '#E74C3C', 1: '#2ECC71'})

# Normal points
normal_idx = df['anomaly_flag'] == 1
atrisk_idx = df['anomaly_flag'] == -1

plt.scatter(X_pca[normal_idx, 0], X_pca[normal_idx, 1],
            c='#2ECC71', alpha=0.3, s=12, label=f'Normal ({normal_idx.sum()})')
plt.scatter(X_pca[atrisk_idx, 0], X_pca[atrisk_idx, 1],
            c='#E74C3C', alpha=0.6, s=20, label=f'AT-RISK ({atrisk_idx.sum()})', marker='x')

plt.title('PCA 2D View — Normal vs AT-RISK Students', fontsize=14, fontweight='bold')
plt.xlabel(f'PCA Component 1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
plt.ylabel(f'PCA Component 2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
plt.legend(fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(ASSETS, 'anomaly_02_pca_2d.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✅ PCA plot saved!')

In [ ]:
# ── CELL 8 ── PLOT 3: Feature-wise Comparison ─────────────────
key_features = [
    'Curricular units 1st sem (approved)',
    'Curricular units 2nd sem (approved)',
    'Curricular units 1st sem (grade)',
    'Curricular units 2nd sem (grade)',
    'Admission grade',
    'Age at enrollment'
]
key_features = [f for f in key_features if f in df.columns]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for i, feat in enumerate(key_features):
    normal_vals = df[df['anomaly_flag'] == 1][feat]
    atrisk_vals = df[df['anomaly_flag'] == -1][feat]

    axes[i].hist(normal_vals, bins=25, alpha=0.6, color='#2ECC71', label='Normal')
    axes[i].hist(atrisk_vals, bins=25, alpha=0.6, color='#E74C3C', label='AT-RISK')
    axes[i].set_title(feat[:40], fontsize=10, fontweight='bold')
    axes[i].set_ylabel('Count')
    axes[i].legend(fontsize=8)

plt.suptitle('Feature Distribution: Normal vs AT-RISK Students', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(ASSETS, 'anomaly_03_feature_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✅ Feature comparison plot saved!')

In [ ]:
# ── CELL 9 ── PLOT 4: Risk Heatmap (Top 20 AT-RISK) ───────────
# Show top 20 most anomalous students as a heatmap
top_atrisk = df[df['anomaly_flag'] == -1].nsmallest(20, 'anomaly_score')

heatmap_features = [
    'Age at enrollment',
    'Curricular units 1st sem (approved)',
    'Curricular units 2nd sem (approved)',
    'Curricular units 1st sem (grade)',
    'Curricular units 2nd sem (grade)',
    'Scholarship holder',
    'Debtor',
    'Tuition fees up to date'
]
heatmap_features = [f for f in heatmap_features if f in df.columns]

heatmap_data = top_atrisk[heatmap_features].reset_index(drop=True)
heatmap_norm = (heatmap_data - heatmap_data.min()) / (heatmap_data.max() - heatmap_data.min() + 1e-6)

plt.figure(figsize=(14, 7))
sns.heatmap(heatmap_norm.T, annot=heatmap_data.T, fmt='.1f',
            cmap='RdYlGn', linewidths=0.5,
            xticklabels=[f'S{i+1}' for i in range(20)],
            yticklabels=[f[:30] for f in heatmap_features],
            cbar_kws={'label': 'Normalized Value (0=low, 1=high)'})
plt.title('Top 20 Most AT-RISK Students — Feature Heatmap', fontsize=13, fontweight='bold')
plt.xlabel('Student (ranked by anomaly score)')
plt.tight_layout()
plt.savefig(os.path.join(ASSETS, 'anomaly_04_risk_heatmap.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✅ Risk heatmap saved!')

In [ ]:
# ── CELL 10 ── Save Model & Results ───────────────────────────
# Save Isolation Forest model
joblib.dump(iso_forest, os.path.join(MODELS, 'isolation_forest.pkl'))
joblib.dump(scaler,     os.path.join(MODELS, 'anomaly_scaler.pkl'))
joblib.dump(FEATURES,   os.path.join(MODELS, 'anomaly_features.pkl'))

# Save flagged students to CSV
atrisk_students = df[df['anomaly_flag'] == -1][FEATURES + ['Target', 'anomaly_score']]
atrisk_students = atrisk_students.sort_values('anomaly_score')
atrisk_students.to_csv(
    os.path.join(BASE, 'data', 'processed', 'at_risk_students.csv'), index=False
)

print('✅ Models saved!')
print(f'   → models/isolation_forest.pkl')
print(f'   → models/anomaly_scaler.pkl')
print(f'   → models/anomaly_features.pkl')
print(f'   → data/processed/at_risk_students.csv')

print('\n' + '='*55)
print('MODULE 1 COMPLETE ✅')
print('='*55)
print(f'Total students       : {len(df)}')
print(f'AT-RISK flagged      : {n_atrisk} ({n_atrisk/len(df)*100:.1f}%)')
print(f'Normal               : {n_normal} ({n_normal/len(df)*100:.1f}%)')
print(f'Actual dropouts caught: {dropout_in_atrisk} ({pct:.1f}% of AT-RISK)')
print('\n🚀 Next: Module 2 — Dropout Risk Classifier (04_dropout_classifier.ipynb)')